In [0]:
SELECT * FROM samples.tpch.orders LIMIT 10;

o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment
11396166,179329,O,187683.10,1996-06-22,4-NOT SPECIFIED,Clerk#000004689,0,"ep fluffily regular packages. regular, final courts ag"
11396167,473245,F,117554.58,1994-07-20,1-URGENT,Clerk#000001856,0,efully after the carefully express packages; final r
11396192,77549,O,47611.56,1996-01-03,3-MEDIUM,Clerk#000001450,0,es. regular tithes poach careful
11396193,610651,F,121869.26,1994-11-29,5-LOW,Clerk#000003696,0,"are furiously along the bold, even ideas. even, final pinto beans"
11396194,193169,O,241066.47,1998-03-21,4-NOT SPECIFIED,Clerk#000002476,0,lar instructions use slyly carefully furious instructions. quickly
11396195,580097,O,59578.69,1995-09-14,1-URGENT,Clerk#000003093,0,ts sleep fluffily. furiously eve
11396196,432472,F,186262.50,1992-10-15,4-NOT SPECIFIED,Clerk#000000132,0,ses-- furiously bold deposits impr
11396197,287354,F,182596.64,1993-09-19,5-LOW,Clerk#000002699,0,ructions. even packages alongside of the furiously ironic accounts slee
11396198,299479,F,98992.78,1994-12-29,1-URGENT,Clerk#000001501,0,"ages. thin, regular accounts nag quickly against the pinto beans. requests wak"
11396199,91625,O,336890.16,1996-12-25,3-MEDIUM,Clerk#000004546,0,cies. blithely final deposits boost carefu


In [0]:
SELECT
  COUNT(*)                                   AS total_orders,
  COUNT(DISTINCT o_orderkey)                 AS unique_order_keys,
  COUNT(*) - COUNT(DISTINCT o_orderkey)      AS duplicate_keys,
  SUM(CASE WHEN o_totalprice IS NULL THEN 1 ELSE 0 END) AS null_prices,
  SUM(CASE WHEN c.c_custkey IS NULL THEN 1 ELSE 0 END)  AS orphan_orders
FROM samples.tpch.orders o
LEFT JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey;

total_orders,unique_order_keys,duplicate_keys,null_prices,orphan_orders
7500000,7500000,0,0,0


In [0]:
SELECT n.n_name AS nation, c.c_mktsegment AS segment,
       COUNT(o.o_orderkey) AS orders, ROUND(SUM(o.o_totalprice), 2) AS revenue
FROM samples.tpch.orders o
JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation   n ON c.c_nationkey = n.n_nationkey
GROUP BY n.n_name, c.c_mktsegment
ORDER BY revenue DESC;

nation,segment,orders,revenue
INDONESIA,FURNITURE,62650,9492791103.08
FRANCE,HOUSEHOLD,62297,9463750801.88
FRANCE,BUILDING,62020,9390572006.71
UNITED KINGDOM,AUTOMOBILE,62030,9378806704.66
JORDAN,BUILDING,61862,9360011661.45
INDONESIA,AUTOMOBILE,61909,9349021901.73
ETHIOPIA,AUTOMOBILE,61954,9345514850.74
IRAQ,AUTOMOBILE,61657,9318101732.95
INDIA,MACHINERY,61681,9311457194.73
GERMANY,FURNITURE,61635,9309531754.06


Databricks visualization. Run in Databricks to view.

In [0]:
WITH cust_rev AS (
  SELECT n.n_name AS nation, c.c_name AS customer, SUM(o.o_totalprice) AS revenue
  FROM samples.tpch.orders o
  JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
  JOIN samples.tpch.nation   n ON c.c_nationkey = n.n_nationkey
  GROUP BY n.n_name, c.c_name
)
SELECT * FROM (
  SELECT *, RANK() OVER (PARTITION BY nation ORDER BY revenue DESC) AS rnk FROM cust_rev
) WHERE rnk <= 3
ORDER BY nation, rnk;

nation,customer,revenue,rnk
ALGERIA,Customer#000365485,6326403.42,1
ALGERIA,Customer#000170077,6248989.88,2
ALGERIA,Customer#000125296,6063998.30,3
ARGENTINA,Customer#000030652,6229541.91,1
ARGENTINA,Customer#000582538,6076442.46,2
ARGENTINA,Customer#000702157,5921972.73,3
BRAZIL,Customer#000172696,6326976.13,1
BRAZIL,Customer#000020551,6191793.22,2
BRAZIL,Customer#000117991,6090944.63,3
CANADA,Customer#000181060,6464706.39,1


In [0]:
%python
from pyspark.sql import functions as F

orders = spark.table("samples.tpch.orders")
clean = (orders
         .dropDuplicates(["o_orderkey"])
         .filter(F.col("o_totalprice").isNotNull())
         .withColumn("order_year", F.year("o_orderdate")))

clean.write.mode("overwrite").saveAsTable("workspace.default.orders_clean")
print(orders.count(), "rows in ->", clean.count(), "rows out")

7500000 rows in -> 7500000 rows out


In [0]:
SELECT order_year, COUNT(*) AS orders, ROUND(SUM(o_totalprice), 2) AS revenue
FROM workspace.default.orders_clean
GROUP BY order_year
ORDER BY order_year;

order_year,orders,revenue
1992,1139873,172177707475.98
1993,1138893,172229906369.71
1994,1137944,172079589599.59
1995,1136308,171632673720.20
1996,1141556,172605588830.66
1997,1137325,171861381581.98
1998,668101,100852367668.13
